In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_date

# Day 1 data
data_day1 = [
    (101, "Rahul", "Mumbai"),
    (102, "Amit", "Pune")
]
columns = ["customer_id", "name", "city"]

bronze_df = spark.createDataFrame(data_day1, columns)

In [0]:
bronze_df.write.format("delta").mode("overwrite").saveAsTable("bronze_customers")

In [0]:
silver_df = (
    spark.table("bronze_customers")
)

silver_df.write.format("delta").mode("overwrite").saveAsTable("silver_customers")


In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold_customer (
    customer_id INT,
    name STRING,
    city STRING,
    start_date DATE,
    end_date DATE,
    is_current BOOLEAN
)
USING DELTA;


In [0]:
%sql
INSERT INTO gold_customer
SELECT
    customer_id,
    name,
    city,
    current_date() AS start_date,
    NULL AS end_date,
    true AS is_current
FROM silver_customers;

In [0]:
data_day2 = [
    (101, "Rahul", "Delhi"),   # Changed city
    (102, "Amit", "Pune"),     # Same
    (103, "Neha", "Jaipur"),    # New customer
    (101,"Rahul","Delhi")
]

silver_day2_df = spark.createDataFrame(data_day2, columns)


In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(spark, "gold_customer")

deltaTable.alias("target").merge(
    silver_day2_df.alias("source"),
    condition="""
        target.customer_id = source.customer_id
        AND target.is_current = true
    """
).whenMatchedUpdate(
    condition="""
        target.name <> source.name
        OR target.city <> source.city
    """,
    set={
        "end_date": "current_date()",
        "is_current": "false"
    }
).execute()

deltaTable.alias("target").merge(
    silver_day2_df.alias("source"),
    condition="""
        target.customer_id = source.customer_id
        AND target.is_current = true
    """
).whenNotMatchedInsert(
    values={
        "customer_id": "source.customer_id",
        "name": "source.name",
        "city": "source.city",
        "start_date": "current_date()",
        "end_date": "cast(null as date)",
        "is_current": "true"
    }
).execute()


In [0]:
%sql
SELECT * FROM gold_customer;
